# 01 — Data Collection


## 1 — Bootstrap


In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

import config
from collect import (collect_all_prices, collect_news, collect_prices,
                     save_news, verify_all_raw)

config.ensure_dirs()
pd.set_option('display.width', 140)

print(f'Registered stocks: {config.list_stocks()}')


## 2 — Current state of the raw data


In [ ]:
status = verify_all_raw()
print()
print(status[['stock', 'ok', 'rows', 'start', 'end', 'years',
              'usable_for_horizon_3']].to_string(index=False))


## 3 — Download prices for every registered stock

> [!WARNING]
> **Curated Data Protection**: The CSV files in `data/raw/` contain historical price data expanded back to 2000 (or listing date) wherever available, with curated cleaning for holiday filler rows and corporate demergers (e.g. ADANIENT 2015 spin-offs and VEDL restructuring via `scripts/fix_raw_data.py`).
>
> - By default, `overwrite=False` skips existing raw files to preserve these curated adjustments.
> - If you ever run `collect_all_prices(overwrite=True)`, raw Yahoo Finance data will replace these files. You **must** immediately re-run `python scripts/fix_raw_data.py` to reapply the demerger and holiday adjustments.


In [ ]:
# overwrite=False skips stocks whose CSV already exists (preserves curated historical data).
# Set overwrite=True only if you explicitly wish to refresh all raw data, followed by scripts/fix_raw_data.py.

paths = collect_all_prices(overwrite=False)
print()
for key, path in paths.items():
    print(f'  {key:12s} {path.name}')


## 4 — Download a single stock, or extend its history

To download a newly registered stock or refresh an individual ticker without touching others, call `collect_prices(stock_key)`.


In [ ]:
# Example: download or refresh a single stock without touching the others.
# (Note: mature project stocks like RELIANCE, TCS, INFY, HDFCBANK already span
# 2000-01-01 to 2026. Do not truncate their date ranges.)

# Example call for an individual stock:
# collect_prices('TCS', start='2004-08-25', end='2026-09-01', overwrite=False)

print("Uncomment and adjust the call above to fetch or refresh a specific stock.")


## 5 — Re-verify after downloading


In [ ]:
status = verify_all_raw()
print()
print(status[['stock', 'ok', 'rows', 'start', 'end', 'years',
              'usable_for_horizon_3']].to_string(index=False))
print()
print('Next: run notebooks/00_setup_and_build.ipynb to rebuild datasets.')


## 6 — Live News Collection (Optional, Live Dashboard Inference Only)

### Setup Instructions
To use the optional live news collector, copy the template `.env.example` to `.env` in the project root:
```bash
cp .env.example .env
```
Then edit `.env` to include your free API key from [newsapi.org](https://newsapi.org/register):
```
NEWSAPI_KEY=your_key_here
```
(The `.env` file is excluded in `.gitignore` and must never be committed.)

### Live Inference vs. Historical Training Archives
- **Live Inference**: The free NewsAPI tier provides headlines from roughly the last 30 days. This is designed for real-time inference in the dashboard ("Why did it move?" recent days), **not** for building a model training corpus.
- **Historical Training Corpus**: Out-of-sample model training uses the curated multi-decade historical news archives in `sentiment_study/` (Times of India 2001–2022 with 276k headlines, NYT 1990–2025, FNSPID, and 78,467 NSE corporate announcements). Each stock's targeted sentiment is stored in `data/processed/<STOCK>_company_sentiment.csv`.


In [ ]:
# Fetch recent headlines from NewsAPI (live inference only)
try:
    news = collect_news('TCS', days_back=28)
    display(news.head())
except ValueError as exc:
    print(exc)
except Exception as exc:
    print(f"Live news collection unavailable: {exc}")


## 7 — Save collected live news


In [ ]:
# Save live news if successfully retrieved
if 'news' in locals() and isinstance(news, pd.DataFrame) and not news.empty:
    # save_news(news, 'TCS')
    print('Uncomment to save. This writes to data/raw/TCS_news_raw.csv.')
else:
    print('No live news collected in this session. Configure NEWSAPI_KEY in .env to enable.')
